## **AMR Interpolation**

Import relevant libraries:

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import struct

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

Define some functions that will be useful:

In [11]:
def split_data(x_data, y_data, val_split):

    train_split = 1 - val_split
    train_size = int(x_data.shape[0] * train_split)

    X_train = x_data[:train_size, :, :, :]
    X_val = x_data[train_size:, :, :, :]

    y_train = y_data[:train_size, :, :, :]
    y_val = y_data[train_size:, :, :, :]

    return (X_train, y_train), (X_val, y_val)

#### Data reading/formatting

In [59]:
aData = np.empty((4608,8,8,8))
cData = np.empty((4608,8,8,8))
bData = np.empty((4608,8,8,8))

with open("data.raw", "rb") as file:
    for n in range(4608):
        for k in range(8):
            for j in range(8):
                for i in range(8):
                    aData[n,i,j,k] = struct.unpack('<f', file.read(4))[0]         

        for k in range(8):
            for j in range(8):
                for i in range(8):
                    cData[n,i,j,k] = struct.unpack('<f', file.read(4))[0]

        for k in range(8):
            for j in range(8):
                for i in range(8):
                    bData[n,i,j,k] = struct.unpack('<f', file.read(4))[0]  

# for n in range(4608):
#     for k in range(8):
#         for j in range(8):
#             for i in range(8):
#                 if aData[n,i,j,k] != cData[n,i,j,k]:
#                     print("There's one")
#                 if aData[n,i,j,k] != bData[n,i,j,k]:
#                     print("There's one")
#                 if cData[n,i,j,k] != bData[n,i,j,k]:
#                     print("There's one")
                    
print(type(aData[1233,0,2,4]))
print(cData[1233,0,2,4])
print(bData[1233,0,2,4])

validation_split = 0.1

aData = torch.from_numpy(aData)
aData = aData.float()
cData = torch.from_numpy(cData)
cData = cData.float()
X_full = torch.cat((aData, cData), dim=1)

y_full = torch.from_numpy(bData).float()

(X_train, y_train), (X_val, y_val) = split_data(X_full, y_full, validation_split)

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

class CombustionData(Dataset):
    def __init__(self, X_full, y_full):
        super().__init__()
        X_full = X_full
        y_full = y_full
    
    def __len__(self):
        return len(X_full)

    def __getitem__(self, idx):
        input = X_full[idx]
        output = y_full[idx]
        # sample = {'input': input, 'output': output}
        return input, output

<class 'numpy.float64'>
-11592927232.0
-11691096064.0
torch.Size([4147, 16, 8, 8])
torch.Size([4147, 8, 8, 8])
torch.Size([461, 16, 8, 8])
torch.Size([461, 8, 8, 8])


#### Define Models

In [79]:
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.1),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.disc(x)

class Generator(nn.Module):
    def __init__(self, n_dim, input_dim):
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(n_dim, 1024),
            nn.LeakyReLU(0.1),
            nn.Linear(1024, input_dim),
            nn.Tanh()
        )

    def forward(self, x):
        return self.gen(x)

#### Hyperparameters

In [80]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)
lr = 1e-4
n_dim = 32
input_dim = 16 * 8 * 8
batch_size = 32
num_epochs = 100

Running on cuda


#### Initializations

In [81]:
disc = Discriminator(input_dim).to(device)
gen = Generator(n_dim, input_dim).to(device)

fixed_noise = torch.randn((batch_size, n_dim)).to(device)

# transform = transforms.Normalize((0.5), (0.5))

dataset = CombustionData(X_full, y_full)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

opt_disc = optim.Adam(disc.parameters(), lr=lr)
opt_gen = optim.Adam(gen.parameters(), lr=lr)

criterion = nn.BCELoss()

#### Train models

In [82]:
for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, 1024).to(device)
        batch_size = real.shape[0]

        # Train discriminator: max log(D(real)) + log(1 - D(G(z))
        noise = torch.randn((batch_size, n_dim)).to(device)
        fake = gen(noise)
        disc_real = disc(real).view(-1)
        lossD_real = criterion(disc_real, torch.ones_like(disc_real)) # this is the log(D(real)) term
        disc_fake = disc(fake).view(-1)
        lossD_fake = criterion(disc_fake, torch.zeros_like(disc_fake)) # this is the log(1 - D(G(z)) term
        lossD = (lossD_real + lossD_fake) / 2
        disc.zero_grad()
        lossD.backward(retain_graph=True)
        opt_disc.step()

        # Train generator: min log(1 - D(G(z)) <==> max log(D(g(z))
        output = disc(fake).view(-1)
        lossG = criterion(output, torch.ones_like(output))
        gen.zero_grad()
        lossG.backward()
        opt_gen.step()

        if batch_idx == 0:
            print(
                f"Epoch [{epoch}/{num_epochs}] \ "
                f"Loss D: {lossD:.4f}, Loss G: {lossG:4f}"
            )

<>:27: SyntaxWarning: invalid escape sequence '\ '
<>:27: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_3257913/2281181572.py:27: SyntaxWarning: invalid escape sequence '\ '
  f"Epoch [{epoch}/{num_epochs}] \ "


Epoch [0/100] \ Loss D: 0.3489, Loss G: 0.722487
Epoch [1/100] \ Loss D: 37.5855, Loss G: 2.202990
Epoch [2/100] \ Loss D: 50.0181, Loss G: 3.507443
Epoch [3/100] \ Loss D: 50.0427, Loss G: 2.729986
Epoch [4/100] \ Loss D: 50.0339, Loss G: 2.956535
Epoch [5/100] \ Loss D: 50.0283, Loss G: 3.140791
Epoch [6/100] \ Loss D: 50.0486, Loss G: 2.635719
Epoch [7/100] \ Loss D: 50.0361, Loss G: 2.867649
Epoch [8/100] \ Loss D: 50.0795, Loss G: 2.138096
Epoch [9/100] \ Loss D: 50.0522, Loss G: 2.430867
Epoch [10/100] \ Loss D: 50.0471, Loss G: 2.576684
Epoch [11/100] \ Loss D: 50.0410, Loss G: 2.601505
Epoch [12/100] \ Loss D: 50.0318, Loss G: 2.921813
Epoch [13/100] \ Loss D: 50.0244, Loss G: 3.141006
Epoch [14/100] \ Loss D: 50.0160, Loss G: 3.511741
Epoch [15/100] \ Loss D: 50.0114, Loss G: 3.888588
Epoch [16/100] \ Loss D: 50.0095, Loss G: 4.107234
Epoch [17/100] \ Loss D: 50.0068, Loss G: 4.448405
Epoch [18/100] \ Loss D: 50.0057, Loss G: 4.616887
Epoch [19/100] \ Loss D: 50.0048, Loss G: 